In [1]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=500, noise=0.3, random_state=67)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=67)

In [3]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.neighbors import KNeighborsClassifier

from ml_from_scratch.preprocessing import StandardScaler
from ml_from_scratch.knn import KNNClassifier, pairwise_distances
from ml_from_scratch.model_selection import k_fold_indices, cross_val_score
from ml_from_scratch.metrics import accuracy

# data: breast cancer, 1 = malignant, same split as before
data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, random_state=42, stratify=y
)

# scale with YOUR scaler: fit on train only, reuse on test
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 1. distances: hand example
A = np.array([[0, 0], [1, 1]])
B = np.array([[3, 4], [1, 1], [0, 1]])
print("distances ok:     ", np.allclose(pairwise_distances(A, B), [[5, np.sqrt(2), 1], [np.sqrt(13), 0, 1]]))

# 2. distances: same as scikit-learn on real data
print("matches sklearn:  ", np.allclose(pairwise_distances(X_test_s, X_train_s),
                                        euclidean_distances(X_test_s, X_train_s), atol=1e-8))

# 3. tiny k-NN example: expected [0, 1, 0.667]
Xt = np.array([[0], [1], [2], [10], [11], [12]])
yt = np.array([0, 0, 0, 1, 1, 1])
print("tiny proba:       ", KNNClassifier(k=3).fit(Xt, yt).predict_prob(np.array([[1.5], [10.5], [6.1]])).round(3))

# 4. k = 1 on the training set: expected 1.0
m1 = KNNClassifier(k=1).fit(X_train_s, y_train)
print("k=1 train acc:    ", accuracy(y_train, m1.predict(X_train_s)))

# 5. same predictions and probabilities as scikit-learn, k = 5
mine = KNNClassifier(k=5).fit(X_train_s, y_train)
sk = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
print("same predictions: ", (mine.predict(X_test_s) == sk.predict(X_test_s)).all())
print("max proba diff:   ", np.abs(mine.predict_prob(X_test_s) - sk.predict_proba(X_test_s)[:, 1]).max())

# 6. test accuracy, k = 5: expected about 0.956
print("k=5 test acc:     ", accuracy(y_test, mine.predict(X_test_s)))

# 7. folds: 5 pairs, validation sets cover 0..9 exactly once, no overlap
folds = k_fold_indices(10, 5)
all_val = np.sort(np.concatenate([v for _, v in folds]))
no_overlap = all(len(np.intersect1d(t, v)) == 0 for t, v in folds)
print("folds ok:         ", len(folds) == 5 and np.array_equal(all_val, np.arange(10)) and no_overlap)
print("fold sizes (455): ", [len(v) for _, v in k_fold_indices(455, 5)])

# 8. cross-validation: 5 scores, mean about 0.96
scores = np.array(cross_val_score(KNNClassifier(k=5), X_train_s, y_train))
print("cv scores:        ", scores.round(3), "mean", scores.mean().round(3))

distances ok:      True
matches sklearn:   True
tiny proba:        [0.    1.    0.667]
k=1 train acc:     1.0
same predictions:  True
max proba diff:    0.0
k=5 test acc:      0.956140350877193
folds ok:          True
fold sizes (455):  [91, 91, 91, 91, 91]
cv scores:         [0.967 0.978 0.956 0.956 0.967] mean 0.965
